In [1]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer

In [2]:
# Using a single LLM for both summary and answering to avoid KeyError
model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
embedder = SentenceTransformer('all-MiniLM-L6-v2')

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
document = """
Climate change refers to long-term shifts in temperatures and weather patterns on Earth.
These changes may occur naturally, but in recent decades human activities have been the main driver.
Burning fossil fuels such as coal, oil, and gas releases large amounts of carbon dioxide into the atmosphere.
This increase in greenhouse gases traps heat and causes the planet's temperature to rise.

Rising global temperatures lead to many environmental changes.
Glaciers and polar ice caps are melting faster than before, which contributes to rising sea levels.
Higher sea levels can cause flooding in coastal cities and threaten communities living near the ocean.
Climate change also affects weather patterns, making extreme weather events like hurricanes, droughts, and heatwaves more frequent and severe.

The impact of climate change is not limited to the environment.
It also affects agriculture, water supplies, and human health.
Farmers may struggle with unpredictable rainfall and longer droughts, which can reduce crop yields.
Water shortages may occur in regions that depend on glaciers or seasonal rainfall.
Additionally, heatwaves and air pollution can increase health risks for vulnerable populations.

To reduce the effects of climate change, many countries are investing in renewable energy sources such as solar, wind, and hydropower.
These energy sources produce little or no greenhouse gas emissions.
Governments and organizations are also promoting energy efficiency, sustainable transportation, and reforestation projects.
By reducing emissions and protecting natural ecosystems, societies can help slow the pace of climate change and protect the planet for future generations.
"""



> Add blockquote



In [4]:
# 3. Split into chunks (~30 words)
def chunk_text(text, chunk_size=30):
    words = text.split()
    return [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

chunks = chunk_text(document)

In [5]:
def split_into_chunks(text, chunk_size=30):
    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)

    return chunks

In [6]:
chunks = split_into_chunks(document)
print(chunks)

['Climate change refers to long-term shifts in temperatures and weather patterns on Earth. These changes may occur naturally, but in recent decades human activities have been the main driver. Burning', "fossil fuels such as coal, oil, and gas releases large amounts of carbon dioxide into the atmosphere. This increase in greenhouse gases traps heat and causes the planet's temperature to", 'rise. Rising global temperatures lead to many environmental changes. Glaciers and polar ice caps are melting faster than before, which contributes to rising sea levels. Higher sea levels can cause', 'flooding in coastal cities and threaten communities living near the ocean. Climate change also affects weather patterns, making extreme weather events like hurricanes, droughts, and heatwaves more frequent and severe.', 'The impact of climate change is not limited to the environment. It also affects agriculture, water supplies, and human health. Farmers may struggle with unpredictable rainfall and longer 

In [7]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
chunk_embeddings = embed_model.encode(chunks)

In [10]:
question = input("\nAsk your question: ")


Ask your question: How does climate change affect agriculture?


In [11]:
question_embedding = embed_model.encode([question])

In [12]:
similarities = cosine_similarity(question_embedding, chunk_embeddings)[0]

In [13]:
top_indices = np.argsort(similarities)[-2:][::-1]

top_chunks = [chunks[i] for i in top_indices]

In [14]:
context = " ".join(top_chunks)

print("\nRetrieved Context:\n")
print(context)


Retrieved Context:

The impact of climate change is not limited to the environment. It also affects agriculture, water supplies, and human health. Farmers may struggle with unpredictable rainfall and longer droughts, which can reduce crop yields. Water shortages may occur in regions that depend on glaciers or seasonal rainfall. Additionally, heatwaves and air pollution can increase health risks for vulnerable populations. To


In [15]:
model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(model_name)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [16]:
prompt = f"""
Use the following context to answer the question.

Context:
{context}

Question:
{question}

Answer:
"""

result = generator(
    prompt,
    max_length=200,
    num_return_sequences=1
)

print("\nGenerated Answer:\n")
print(result[0]["generated_text"])

Passing `generation_config` together with generation-related arguments=({'max_length', 'num_return_sequences'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Generated Answer:


Use the following context to answer the question.

Context:
The impact of climate change is not limited to the environment. It also affects agriculture, water supplies, and human health. Farmers may struggle with unpredictable rainfall and longer droughts, which can reduce crop yields. Water shortages may occur in regions that depend on glaciers or seasonal rainfall. Additionally, heatwaves and air pollution can increase health risks for vulnerable populations. To

Question:
How does climate change affect agriculture?

Answer:
Climate change affects agriculture. When a particular country and region experiences extreme weather, it is important to prepare for extreme weather. Climate change impacts agricultural productivity and energy availability. However, if a particular country is experiencing extreme weather, it is important to prepare for extreme weather.
Question:
What is the impact of weather events or climate change on crop yields?
Answer:
Climate change impa